In [2]:
!pip install keras.tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 5.2 MB/s eta 0:00:00


In [3]:
import keras_tuner as kt
print(kt.__version__)

1.4.8


In [4]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

In [5]:
fashion_amnist = keras.datasets.fashion_mnist

In [6]:
(train_images,train_lables),(test_images,test_lables) = fashion_amnist.load_data()

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [7]:
train_images=train_images/255.0
test_images=test_images/255.0

In [8]:
train_images = train_images.reshape(len(train_images), 28, 28, 1)

test_images = test_images.reshape(len(test_images), 28, 28, 1)

In [9]:
from tensorflow import keras

def build_model(hp):

    model = keras.Sequential([

        keras.layers.Conv2D(
            filters=hp.Int(
                'conv_1_filter',
                min_value=32,
                max_value=128,
                step=16
            ),
            kernel_size=hp.Choice(
                'conv_1_kernel',
                values=[3, 5]
            ),
            activation='relu',
            input_shape=(28, 28, 1)
        ),

        keras.layers.Conv2D(
            filters=hp.Int(
                'conv_2_filter',
                min_value=32,
                max_value=64,
                step=16
            ),
            kernel_size=hp.Choice(
                'conv_2_kernel',
                values=[3, 5]
            ),
            activation='relu'
        ),

        keras.layers.Flatten(),

        keras.layers.Dense(
            units=hp.Int(
                'dense_1_units',
                min_value=32,
                max_value=128,
                step=16
            ),
            activation='relu'
        ),

        keras.layers.Dense(10, activation='softmax')

    ])

    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice(
                'learning_rate',
                values=[1e-2, 1e-3]
            )
        ),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [10]:
from keras_tuner import RandomSearch
from keras_tuner.engine.hyperparameters import HyperParameters

In [11]:
tuner_search=RandomSearch(build_model,objective='val_accuracy',max_trials=5 ,directory='output',project_name="Mnist Fashion")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [12]:
tuner_search.search(train_images,train_lables,epochs=3,validation_split=0.1)

Trial 5 Complete [00h 00m 30s]
val_accuracy: 0.8818333148956299

Best val_accuracy So Far: 0.9053333401679993
Total elapsed time: 00h 02m 47s


In [13]:
model=tuner_search.get_best_models(num_models=1)[0]
model

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


<Sequential name=sequential, built=True>

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 48)     │           480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 22, 22, 48)     │        57,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 23232)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 96)             │     2,230,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           970 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,289,466 (8.73 MB)

 Trainable params: 2,289,466 (8.73 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(train_images, train_lables, epochs=10 , validation_split=0.1 ,initial_epoch=3)


Epoch 4/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9491 - loss: 0.1359 - val_accuracy: 0.9075 - val_loss: 0.2641
Epoch 5/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9667 - loss: 0.0908 - val_accuracy: 0.9072 - val_loss: 0.2918
Epoch 6/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9768 - loss: 0.0629 - val_accuracy: 0.9055 - val_loss: 0.3225
Epoch 7/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9828 - loss: 0.0464 - val_accuracy: 0.9087 - val_loss: 0.4210
Epoch 8/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9874 - loss: 0.0343 - val_accuracy: 0.9087 - val_loss: 0.5002
Epoch 9/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9897 - loss: 0.0272 - val_accuracy: 0.9152 - val_loss: 0.4887
Epoch 10/10
1688/1688 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.9912 - loss: 0.0251 - val_accuracy: 0.9032 - val_loss: 0.5018


## this is how a model is saved in kerasm